In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import col


In [0]:
dbutils.widgets.text("catalog", "")
catalog = dbutils.widgets.get("catalog")

In [0]:
city_df = spark.read.format("delta").table(f"{catalog}.bronze_transport.city")
city_df.display()

In [0]:
city_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.silver_transport.city_silver")

In [0]:
trips_df = spark.read.format("delta").table(f"{catalog}.bronze_transport.trips_bronze")
trips_df.display()

In [0]:
df_silver = trips_df.select(
    col("trip_id").alias("id"),
    to_date(col("trip_date"), "dd-MM-yyyy").alias("business_date"),
    col("city_id").alias("city_id"),
    col("passenger_type").alias("passenger_category"),
    col("distance_travelled").alias("distance_kms"),
    col("fare_amount").alias("sale_amt"),
    col("passenger_rating").alias("passenger_rating"),
    col("driver_rating").alias("driver_rating"),
    col("ingest_timestamp").alias("bronze_ingest_timestamp")
)

df_silver = df_silver.withColumn(
    "silver_processed_timestamp",
    current_timestamp()
)

display(df_silver)

In [0]:
trips_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.silver_transport.trips_silver")